# Lines

An `ACLineSegment` connects two nodes. Its impedance can be given in three ways, from simplest to most
detailed:

1. directly on the line, as total r, x and b, for a balanced transmission model,
2. from a per-length catalog entry, a sequence impedance or a phase impedance matrix, shared by every line
   built the same way,
3. from the physical construction: the wires and their spacing on the pole.

## Set up

The CIM profile and unit handling are process-wide environment variables, so set them before importing
anything from CIMantic Graphs. `new_feeder_model` makes an empty model to build into.

In [1]:
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"

from cimtables.units import Qty

from cimbuilder.network import new_feeder_model
from cimbuilder.object_builder import (
    ACLineSegmentBuilder,
    BaseVoltageBuilder,
    OverheadWireInfoBuilder,
    PerLengthPhaseImpedanceBuilder,
    PerLengthSequenceImpedanceBuilder,
    SeriesCompensatorBuilder,
    WireConfigurationBuilder,
    XYSpacingBuilder,
)

network = new_feeder_model(
    authority="https://example.com/models", model="lines", Feeder="fdr_demo"
)
container = network.container
base_voltage_builder = BaseVoltageBuilder(network, container)
for mRID, nominalVoltage in [
    ("bv_115kV", Qty(115, "kV")),
    ("bv_4_16kV", Qty(4.16, "kV")),
]:
    base_voltage_builder.create(mRID=mRID, name=mRID, nominalVoltage=nominalVoltage)

line_builder = ACLineSegmentBuilder(network, container)

## A balanced line

`add_connectivity` gives the line its two Terminals, its length and, with `phases`, one
`ACLineSegmentPhase` per conductor. `add_electrical` sets the positive-sequence impedance of the whole
line, and `add_short_circuit` the zero-sequence impedance.

In [2]:
line = line_builder.create(mRID="line_101_102", name="101-102")
line, terminals, line_phases = line_builder.add_connectivity(
    line,
    node1="bus_101",
    node2="bus_102",
    length=Qty(12, "mi"),
    BaseVoltage="bv_115kV",
)
line_builder.add_electrical(
    line, r=Qty(2.1, "ohm"), x=Qty(9.4, "ohm"), bch=Qty(68, "uS")
)
line_builder.add_short_circuit(line, r0=Qty(6.8, "ohm"), x0=Qty(29, "ohm"))

print([terminal.mRID for terminal in terminals])
print(f"{line.length.to('km'):.2f} km")
print(f"x = {line.x.to('ohm'):g} ohm, x0 = {line.x0.to('ohm'):g} ohm")

['line_101_102_T1', 'line_101_102_T2']
19.31 km
x = 9.4 ohm, x0 = 29 ohm


## Per-length sequence impedance

A `PerLengthSequenceImpedance` holds the impedance per unit length. Every line that refers to it takes
its impedance from it, scaled by the line's own length.

In [3]:
sequence_builder = PerLengthSequenceImpedanceBuilder(network, container)
impedance = sequence_builder.create(mRID="plz_477_acsr", name="477 ACSR")
sequence_builder.add_electrical(
    impedance, r=Qty(0.19, "ohm/mi"), x=Qty(0.6, "ohm/mi"), bch=Qty(5.6, "uS/mi")
)
sequence_builder.add_short_circuit(
    impedance, r0=Qty(0.48, "ohm/mi"), x0=Qty(1.9, "ohm/mi")
)

for number, miles in [(1, 0.5), (2, 1.2)]:
    line = line_builder.create(mRID=f"line_seq_{number}")
    line_builder.add_connectivity(
        line,
        node1=f"n{number}",
        node2=f"n{number + 1}",
        phases="ABC",
        length=Qty(miles, "mi"),
        BaseVoltage="bv_4_16kV",
    )
    line_builder.add_per_length_impedance(line, PerLengthImpedance="plz_477_acsr")

print([line.mRID for line in impedance.ACLineSegments])

['line_seq_1', 'line_seq_2']


## Per-length phase impedance matrix

An unbalanced distribution model needs the full phase impedance matrix. `PerLengthPhaseImpedanceBuilder`
builds it one `PhaseImpedanceData` entry at a time, by row and column, here the lower triangle of a
three-conductor matrix (IEEE 13 node feeder configuration 601, in ohm and µS per mile).

The rows and columns are the line's conductors, in `ACLineSegmentPhase.sequenceNumber` order. The phase
string is in conductor order, so `phases="BAC"` puts phase B on row 1.

In [4]:
R = [[0.3465], [0.1560, 0.3375], [0.1580, 0.1535, 0.3414]]
X = [[1.0179], [0.5017, 1.0478], [0.4236, 0.3849, 1.0348]]
B = [[6.2998], [-1.9958, 5.9597], [-1.2595, -0.7417, 5.6386]]

matrix_builder = PerLengthPhaseImpedanceBuilder(network, container)
matrix = matrix_builder.create(mRID="plz_601", name="601")
matrix_builder.add_electrical(matrix, conductorCount=3)
for row in range(1, 4):
    for column in range(1, row + 1):
        matrix_builder.add_phase_impedance_data_row(
            matrix,
            row=row,
            column=column,
            r=Qty(R[row - 1][column - 1], "ohm/mi"),
            x=Qty(X[row - 1][column - 1], "ohm/mi"),
            b=Qty(B[row - 1][column - 1], "uS/mi"),
        )

line = line_builder.create(mRID="line_650_632", name="650-632")
line, _, line_phases = line_builder.add_connectivity(
    line,
    node1="650",
    node2="632",
    phases="BAC",
    length=Qty(2000, "ft"),
    BaseVoltage="bv_4_16kV",
)
line_builder.add_per_length_impedance(line, PerLengthImpedance="plz_601")

print(len(matrix.PhaseImpedanceData), "matrix entries")
print([(phase.sequenceNumber, phase.phase.value) for phase in line_phases])

6 matrix entries
[(1, 'B'), (2, 'A'), (3, 'C')]


## Wires and spacing

To describe the physical line instead, build its asset catalog:

- an `OverheadWireInfo` (or `ConcentricNeutralCableInfo`, `TapeShieldCableInfo`) per conductor type,
- an `XYSpacing` with the position of each conductor on the pole,
- a `WireConfiguration` that puts a wire in each position.

A line then refers to the `WireConfiguration`, and a tool that computes impedances from geometry (such as
OpenDSS) can work out its phase impedance matrix.

In [5]:
wire_builder = OverheadWireInfoBuilder(network, container)
phase_wire = wire_builder.create(mRID="wire_556500_26_7", name="556,500 26/7 ACSR")
wire_builder.add_asset_info(
    phase_wire,
    material="acsr",
    gmr=Qty(0.031, "ft"),
    radius=Qty(0.927 / 2, "in"),
    rAC50=Qty(0.1859, "ohm/mi"),
    ratedCurrent=Qty(730, "A"),
)
neutral_wire = wire_builder.create(mRID="wire_4_0_6_1", name="4/0 6/1 ACSR")
wire_builder.add_asset_info(
    neutral_wire,
    material="acsr",
    gmr=Qty(0.00814, "ft"),
    radius=Qty(0.563 / 2, "in"),
    rAC50=Qty(0.592, "ohm/mi"),
    ratedCurrent=Qty(340, "A"),
)

spacing_builder = XYSpacingBuilder(network, container)
spacing = spacing_builder.create(mRID="spc_500", name="500")
for sequenceNumber, x, y, kind in [
    (1, 0, 28, "phase"),
    (2, 2.5, 28, "phase"),
    (3, 7, 28, "phase"),
    (4, 4, 24, "neutral"),
]:
    spacing_builder.add_wire_position_row(
        spacing,
        sequenceNumber=sequenceNumber,
        xCoord=Qty(x, "ft"),
        yCoord=Qty(y, "ft"),
        kind=kind,
    )

configuration_builder = WireConfigurationBuilder(network, container)
configuration = configuration_builder.create(
    mRID="wc_601", name="601", numberOfPhases=3, WireSpacing="spc_500"
)
for sequenceNumber in (1, 2, 3):
    configuration_builder.add_wire_sequence_configuration(
        configuration, sequenceNumber=sequenceNumber, WireInfo="wire_556500_26_7"
    )
configuration_builder.add_wire_sequence_configuration(
    configuration, sequenceNumber=4, WireInfo="wire_4_0_6_1"
)

line = line_builder.create(mRID="line_632_671", name="632-671")
line_builder.add_connectivity(
    line,
    node1="632",
    node2="671",
    phases="BACN",
    length=Qty(2000, "ft"),
    BaseVoltage="bv_4_16kV",
)
line_builder.add_wire_configuration(line, WireConfiguration="wc_601")

for wire_sequence in sorted(
    configuration.WireSequence, key=lambda w: w.sequenceNumber
):
    print(wire_sequence.sequenceNumber, wire_sequence.WireInfo.name)

1 556,500 26/7 ACSR
2 556,500 26/7 ACSR
3 556,500 26/7 ACSR
4 4/0 6/1 ACSR


A line whose phases use different wires can instead link each `ACLineSegmentPhase` to its wire, with
`ACLineSegmentPhaseBuilder.add_asset_deployment(phase, WireInfo=...)`. Underground cables use
`ConcentricNeutralCableInfoBuilder` in place of `OverheadWireInfoBuilder`, with the neutral strand data as
well.

## Series compensators

A `SeriesCompensator` is a series capacitor or reactor between two nodes, built the same way as a balanced
line without a length:

In [6]:
compensator_builder = SeriesCompensatorBuilder(network, container)
compensator = compensator_builder.create(mRID="sc_102", name="series cap 102")
compensator_builder.add_connectivity(
    compensator, node1="bus_102", node2="bus_102_sc", BaseVoltage="bv_115kV"
)
compensator_builder.add_electrical(compensator, r=Qty(0, "ohm"), x=Qty(-4.5, "ohm"))
print(f"x = {compensator.x.to('ohm'):g} ohm")

x = -4.5 ohm
